<a href="https://colab.research.google.com/github/Kavishka2401/GPTea/blob/C_1_Yield_Prediction/Machine_Learning_Pipeline/XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Google Drive to get the final dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# IMPORT LIBRARIES
import pandas as pd
import json
import joblib
import numpy as np
from groq import Groq

In [5]:
# LOAD MODEL AND DATA
# Load trained Ridge model pipeline
ridge_model = joblib.load("/content/drive/MyDrive/DSGP_DOCS/Final_Models/ridge_model.pkl")

# Load preprocessed dataset
df = pd.read_csv("/content/drive/MyDrive/DSGP_DOCS/Datasets/df_ridge_preprocessed.csv")

# Load feature names
with open("/content/drive/MyDrive/DSGP_DOCS/Datasets/ridge_features.json", "r") as f:
    features = json.load(f)

In [6]:
# EXTRACT PIPELINE STEPS
imputer = ridge_model.named_steps['imputer']
scaler  = ridge_model.named_steps['scaler']
ridge   = ridge_model.named_steps['ridge']

In [7]:
# PREPROCESS A SAMPLE ROW

# Pick a sample row (last row in dataset)
X = df[features]  # Ensure only feature columns are selected
sample_raw = X.iloc[-1:]

# Impute missing values
X_imputed = imputer.transform(sample_raw)

# Scale features
X_scaled = scaler.transform(X_imputed)

# Compute feature contributions
contributions = X_scaled[0] * ridge.coef_

# Create a DataFrame with feature, value, and contribution
explanation_df = pd.DataFrame({
    "Feature": features,
    "Value": sample_raw.values.flatten(),
    "Contribution": contributions
})

print("=== Feature Contributions for Sample ===")
print(explanation_df)

=== Feature Contributions for Sample ===
                       Feature          Value  Contribution
0                         T200       0.000000     -0.967185
1                        UT925       0.000000     -1.210339
2                         U709   13600.000000      1.754656
3                   Epson Salt       0.000000     -1.037490
4                     Dolomite   21000.000000      4.066805
5                         Urea       0.000000     -0.860133
6                         Zinc       0.000000     -0.670779
7                      Compost     205.000000     -0.160515
8                   BioShakthi       0.000000      0.143175
9                    BioShield       0.000000     -1.807824
10         temperature_2m_mean      18.633333     -1.214987
11   relative_humidity_2m_mean      86.166667      1.385212
12                    rain_sum      47.400000     -2.479399
13           sunshine_duration  208409.870000     -3.561132
14         wind_speed_10m_mean       3.400000      4.370142

In [8]:
# PREPARE TEXT FOR GROQ EXPLANATION

# Create a string showing the important features and their contributions
feature_summary = ""
for idx, row in explanation_df.iterrows():
    feature_summary += f"{row['Feature']} = {row['Value']} (contribution: {row['Contribution']:.2f})\n"

prompt = f"Explain the following features and their contributions to tea yield in simple words:\n\n{feature_summary}"

In [9]:
# SEND TO GROQ AI

# Initialize Groq client
client = Groq(api_key="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")

# Create a chat completion request
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [10]:
# DISPLAY EXPLANATION
print("\n=== Groq AI Explanation ===")
print(response.choices[0].message.content)


=== Groq AI Explanation ===
Let's break down each feature and its contribution to tea yield in simple words:

1. **T200**: No contribution (0.0), but if applied, it would decrease tea yield (-0.97).
2. **UT925**: No contribution (0.0), but if applied, it would significantly decrease tea yield (-1.21).
3. **U709**: High contribution (13600.0) and increases tea yield (1.75).
4. **Epson Salt**: No contribution (0.0), but if applied, it would decrease tea yield (-1.04).
5. **Dolomite**: Very high contribution (21000.0) and significantly increases tea yield (4.07).
6. **Urea**: No contribution (0.0), but if applied, it would decrease tea yield (-0.86).
7. **Zinc**: No contribution (0.0), but if applied, it would slightly decrease tea yield (-0.67).
8. **Compost**: Low contribution (205.0) and slightly decreases tea yield (-0.16).
9. **BioShakthi**: No contribution (0.0), but if applied, it would slightly increase tea yield (0.14).
10. **BioShield**: No contribution (0.0), but if applied, i

DEMO

In [15]:
# LOAD MODEL AND DATA
# Load Ridge model pipeline
ridge_model = joblib.load("/content/drive/MyDrive/DSGP_DOCS/Final_Models/ridge_model.pkl")

# Load dataset (for scaling / reference)
df = pd.read_csv("/content/drive/MyDrive/DSGP_DOCS/Datasets/df_ridge_preprocessed.csv")

# Load feature list
with open("/content/drive/MyDrive/DSGP_DOCS/Datasets/ridge_features.json", "r") as f:
    features = json.load(f)

# Extract pipeline steps
imputer = ridge_model.named_steps['imputer']
scaler  = ridge_model.named_steps['scaler']
ridge   = ridge_model.named_steps['ridge']

# CREATE SAMPLE INPUT
sample_values = {
    "T200": 1200.0,
    "UT925": 0.0,
    "U709": 13600.0,
    "Epson Salt": 0.0,
    "Dolomite": 21000.0,
    "Urea": 0.0,
    "Zinc": 0.0,
    "Compost": 205.0,
    "BioShakthi": 0.0,
    "BioShield": 0.0,
    "temperature_2m_mean": 18.633333,
    "relative_humidity_2m_mean": 86.166667,
    "rain_sum": 47.4,
    "sunshine_duration": 208409.87,
    "wind_speed_10m_mean": 3.4,
    "et0_fao_evapotranspiration": 3.34,
    "YPH_lag1": 188.0,
    "YPH_lag6": 145.0,
    "YPH_lag12": 172.0,
    "UT925_lag1": 0.0,
    "U709_lag1": 0.0,
    "Dolomite_lag1": 0.0
}

# Convert to DataFrame
sample_df = pd.DataFrame([sample_values])

# PREPROCESS SAMPLE
X_imputed = imputer.transform(sample_df)
X_scaled  = scaler.transform(X_imputed)

# PREDICT YIELD
prediction = ridge.predict(X_scaled)
print("Predicted Yield:", prediction[0])

# CALCULATE FEATURE CONTRIBUTIONS
# Multiply scaled inputs by coefficients
contributions = X_scaled[0] * ridge.coef_

# Combine features, raw values, and contributions
explanation_df = pd.DataFrame({
    "Feature": features,
    "Value": sample_df.iloc[0].values,
    "Contribution": contributions
})

print("\n=== Feature Contributions for Sample ===")
print(explanation_df)

# SEND EXPLANATION REQUEST TO GROQ AI

# Categorize features
fertilizers = ["U709", "UT925", "Dolomite", "Urea", "Epson Salt", "Zinc", "Compost", "BioShakthi", "BioShield", "Dolomite_lag1", "U709_lag1", "UT925_lag1"]
weather = ["temperature_2m_mean", "relative_humidity_2m_mean", "rain_sum", "sunshine_duration", "wind_speed_10m_mean", "et0_fao_evapotranspiration"]
past_yield = ["YPH_lag1", "YPH_lag6", "YPH_lag12"]

# Build readable string for Groq
def build_feature_string(df):
    fert_str = "\n".join([f"{f} = {df.loc[df['Feature']==f, 'Value'].values[0]} (contribution: {df.loc[df['Feature']==f, 'Contribution'].values[0]:.2f})" for f in fertilizers])
    weather_str = "\n".join([f"{f} = {df.loc[df['Feature']==f, 'Value'].values[0]} (contribution: {df.loc[df['Feature']==f, 'Contribution'].values[0]:.2f})" for f in weather])
    yield_str = "\n".join([f"{f} = {df.loc[df['Feature']==f, 'Value'].values[0]} (contribution: {df.loc[df['Feature']==f, 'Contribution'].values[0]:.2f})" for f in past_yield])

    return f"Fertilizers applied:\n{fert_str}\n\nWeather/Environment variables:\n{weather_str}\n\nPrevious yields:\n{yield_str}"

features_str = build_feature_string(explanation_df)

prompt = f"Explain the effect of these features on tea yield in simple words, mentioning which are fertilizers, which are weather conditions, and which are past yields:\n\n{features_str}"

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print("\n=== Groq AI Explanation ===")
print(response.choices[0].message.content)


Predicted Yield: 179.19500131344896

=== Feature Contributions for Sample ===
                       Feature          Value  Contribution
0                         T200    1200.000000      5.664939
1                        UT925       0.000000     -1.210339
2                         U709   13600.000000      1.754656
3                   Epson Salt       0.000000     -1.037490
4                     Dolomite   21000.000000      4.066805
5                         Urea       0.000000     -0.860133
6                         Zinc       0.000000     -0.670779
7                      Compost     205.000000     -0.160515
8                   BioShakthi       0.000000      0.143175
9                    BioShield       0.000000     -1.807824
10         temperature_2m_mean      18.633333     -1.214988
11   relative_humidity_2m_mean      86.166667      1.385211
12                    rain_sum      47.400000     -2.479399
13           sunshine_duration  208409.870000     -3.561132
14         wind_speed_